In [ ]:
# Research Paper Visualization & Evaluation Notebook
## Satellite-Based Environment Change Detection

This notebook generates publication-quality figures and tables for:
- Training and validation curves
- Confusion matrix and classification metrics
- ROC and Precision–Recall curves
- Threshold calibration plots
- Model and per-scene comparisons
- Satellite image, ground truth, prediction, and error-map visualizations
- Model complexity and inference-time comparisons

All reported results must come from actual training logs, ground-truth labels, and model predictions. Missing data are reported rather than fabricated.

Recommended output: 300-DPI PNG, PDF, CSV metric tables.


In [ ]:
from pathlib import Path
import json
import warnings
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    jaccard_score,
    accuracy_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Update these paths if your artifacts are stored elsewhere.
PROJECT_DIR = Path("/kaggle/working/mh300_changeformer_tiny_oscd")
OUTPUT_DIR = PROJECT_DIR / "research_paper_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: place exported evaluation predictions here.
# NPZ must contain y_true and y_prob arrays.
PREDICTIONS_NPZ = PROJECT_DIR / "test_predictions.npz"

# Optional: validation predictions used for threshold selection.
VALIDATION_NPZ = PROJECT_DIR / "validation_predictions.npz"

# Optional CSVs to compare multiple trained models or scenes.
MODEL_COMPARISON_CSV = PROJECT_DIR / "model_comparison.csv"
SCENE_METRICS_CSV = PROJECT_DIR / "per_scene_metrics.csv"

DPI = 300
FIGURE_FORMATS = ["png", "pdf"]
DEFAULT_THRESHOLD = 0.5

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": DPI,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.autolayout": True
})

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Project directory exists:", PROJECT_DIR.exists())


In [ ]:
def save_figure(fig, name):
    """Save one figure in publication-quality PNG and PDF formats."""
    for ext in FIGURE_FORMATS:
        path = OUTPUT_DIR / f"{name}.{ext}"
        fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"Saved: {name}.png and {name}.pdf")


def save_table(df, name):
    """Save a result table as CSV."""
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print("Saved table:", path)


def find_csv_files(root):
    if not root.exists():
        return []
    return sorted(root.rglob("*.csv"))


def find_history_csv(root):
    """Find likely training-history CSVs without assuming one exact filename."""
    files = find_csv_files(root)
    preferred = [
        p for p in files
        if any(word in p.name.lower() for word in
               ["history", "training_log", "train_log", "epoch_log"])
    ]
    return preferred


print("CSV files found:", len(find_csv_files(PROJECT_DIR)))
for p in find_history_csv(PROJECT_DIR)[:20]:
    print(p)
